<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-mnist-lenet5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classify MNIST with LeNet5

In this notebook, we'll implement [LeNet5](https://www.researchgate.net/publication/2985446_Gradient-Based_Learning_Applied_to_Document_Recognition), a classic convolutional neural network (CNN) that can classify MNIST digits with over 98% accuracy.


## Setup

Install the required packages:


In [ ]:
!pip install torch
!pip install tsilva-notebook-utils

Set up the configuration we'll use throughout the notebook:


In [ ]:
CONFIG = {
    "seed" : 42,
    "n_epochs" : 5,
    "learning_rate" : 1e-3,
    "batch_size": 64
}

Set the random seed for reproducibility:


In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)  # Set the seed for Python's built-in random number generator
    np.random.seed(seed)  # Set the seed for NumPy's random number generator
    torch.manual_seed(seed)  # Set the seed for PyTorch's CPU random number generator
    torch.cuda.manual_seed(seed)  # Set the seed for PyTorch's CUDA (GPU) random number generator, if GPU is available
    torch.backends.cudnn.deterministic = True  # Force CuDNN (CUDA Deep Neural Network library) to use deterministic algorithms for reproducibility
    torch.backends.cudnn.benchmark = False  # Disable CuDNN's benchmarking feature, which optimizes for speed but may introduce randomness

seed = CONFIG["seed"]
set_seed(seed)

Select the device for training (GPU if available, otherwise CPU):


In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Prepare Dataset

In [ ]:
import multiprocessing
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

def load_data(batch_size=None):
    if batch_size is None: batch_size = CONFIG["batch_size"]

    # Number of subprocesses for data loading (parallel processing)
    # (we max these out because CPU is mostly free,
    # eg: we are not doing data augmentation)
    num_workers = multiprocessing.cpu_count()

    # Whether to copy tensors to CUDA pinned memory
    # before returning them (faster GPU transfer)
    pin_memory = 'cuda' in str(DEVICE)

    # Create the transforms to be applied to the dataset
    transform = transforms.Compose([
        transforms.ToTensor(),  # Convert images to PyTorch tensors with values between 0 and 1
        transforms.Normalize((0.1307,), (0.3081,))  # Normalize by mean (0.1307) and std dev (0.3081) of MNIST dataset
    ])

    # Create training set data loader
    num_cores = multiprocessing.cpu_count()  # Get number of available CPU cores for parallel processing
    train_dataset = torchvision.datasets.MNIST(
        root='./data',  # Directory where dataset will be stored
        train=True,  # Load training split of dataset
        download=True,  # Download dataset if not already present
        transform=transform  # Apply defined transformations to each sample
    )
    train_loader = DataLoader(
        train_dataset,  # The dataset the loader iterates over
        batch_size=batch_size,  # Number of samples per batch
        shuffle=True,  # Randomly shuffle samples each epoch
        num_workers=num_cores,  # Number of subprocesses for data loading (parallel processing)
        pin_memory=pin_memory  # Copy tensors to CUDA pinned memory before returning them (faster GPU transfer)
    )

    # Create test set data loader
    test_dataset = torchvision.datasets.MNIST(
        root='./data',  # Directory where dataset will be stored
        train=False,  # Load test split of dataset
        download=True,  # Download dataset if not already present
        transform=transform  # Apply defined transformations to each sample
    )
    test_loader = DataLoader(
        test_dataset,  # The dataset the loader iterates over
        batch_size=batch_size,  # Number of samples per batch
        shuffle=False,  # Load data sequentially (no shuffling) for consistent evaluation
        num_workers=num_cores,  # Number of subprocesses for data loading (parallel processing)
        pin_memory=pin_memory  # Copy tensors to CUDA pinned memory before returning them (faster GPU transfer)
    )

    # Return dataset loaders
    return train_loader, test_loader

train_loader, test_loader = load_data()

Let's inspect a batch from the training set:


In [ ]:
images, labels = next(iter(train_loader))
images.shape, labels.shape

The `images` tensor has shape `(64, 1, 28, 28)`, representing a batch of 64 grayscale images, each sized 28x28 pixels. The `labels` tensor has shape `(64,)`, containing the class label for each image in the batch.


Let's inspect the labels:

In [ ]:
labels

Each label is the digit (0-9) shown in the corresponding input image.


Let's examine the first row of the first image:


In [ ]:
images[0][0][0] # 1st batch entry, 1st channel, 1st row

All values appear identical because the images are normalized by the transform in the data loader. These values likely represent the black background, which is common along the image edges.


Let's display one of these images along with its label:


In [ ]:
import matplotlib.pyplot as plt
index = 4 # Sample we want to render, change this to load a different result (max index = batch size - 1)
labels[index], plt.imshow(images[index].squeeze(), cmap='gray')

## Build Model

Let's build the model using the exact layer structure described in the original LeNet5 paper:


In [ ]:
import torch
import torch.nn as nn

class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()

        # First convolutional layer (1x28x28 -> 6x24x24)
        # Produces feature maps to detect low-level features like edges.
        # out_size = ((in_size - kernel_size + 2 * padding) / stride)) + 1 = ((28 - 5 + 2 * 0) / 1) + 1 = 26
        self.conv1 = nn.Conv2d(
            in_channels=1,   # Input is a grayscale image (1 channel, unlike 3 for RGB).
            out_channels=6,  # Outputs 6 feature maps.
            kernel_size=5,   # Uses 5x5 filters.
            stride=1,        # Moves filter 1 pixel at a time.
            padding=0        # No padding, reduces size to 28-5+1=24.
        )

        # First pooling layer (6x24x24 -> 6x12x12)
        # Reduces overfitting and computational load while retaining key features.
        # out_size = ((in_size - kernel_size + 2 * padding) / stride)) + 1 = ((24 - 2 + 2 * 0) / 2) + 1 = 12
        self.pool1 = nn.MaxPool2d(
            kernel_size=2,   # Uses 2x2 filters.
            stride=2         # Moves filter 2 pixels at a time.
        )

        # Second convolutional layer (6x12x12 -> 16x8x8)
        # Detects more complex patterns from the previous feature maps.
        # out_size = ((in_size - kernel_size + 2 * padding) / stride)) + 1 = ((12 - 5 + 2 * 0) / 1) + 1 = 8
        self.conv2 = nn.Conv2d(
            in_channels=6,   # Takes 6 feature maps as input.
            out_channels=16, # Outputs 16 feature maps.
            kernel_size=5,   # Uses 5x5 filters.
            stride=1,        # Moves filter 1 pixel at a time.
            padding=0        # No padding
        )

        # Second pooling layer (16x8x8 -> 16x4x4)
        # Further reduces dimensions and extracts dominant features.
        # out_size = ((in_size - kernel_size + 2 * padding) / stride)) + 1 = ((8 - 2 + 2 * 0) / 2) + 1 = 4
        self.pool2 = nn.MaxPool2d(
            kernel_size=2,   # Uses 2x2 filters.
            stride=2         # Moves filter 2 pixels at a time.
        )

        # First fully connected layer (256 -> 120)
        # Combines spatial features into abstract representations for higher-level reasoning.
        self.fc1 = nn.Linear(
            in_features=4*4*16, # Flattened input from 4x4x16=256.
            out_features=120    # Outputs 120 neurons.
        )

        # Second fully connected layer (120 -> 84)
        # Further refines features for classification.
        self.fc2 = nn.Linear(
            in_features=120, # Takes 120 neurons as input.
            out_features=84  # Outputs 84 neurons (arbitrary choice in original design).
        )

        # Output layer (84 -> 10)
        # Produces raw scores for each of the 10 MNIST digit classes.
        self.fc3 = nn.Linear(
            in_features=84,  # Takes 84 neurons as input.
            out_features=10  # Outputs 10 neurons, one per digit.
        )

    def forward(self, x):
        # Apply first convolution + tanh (batch_size, 1, 28, 28) -> (batch_size, 6, 24, 24)
        # Tanh activation squashes values to [-1, 1], as used in original LeNet-5.
        x = torch.tanh(self.conv1(x))

        # Apply first max pooling (batch_size, 6, 24, 24) -> (batch_size, 6, 12, 12)
        # Takes max over 2x2 regions to downsample.
        x = self.pool1(x)

        # Apply second convolution + tanh (batch_size, 6, 12, 12) -> (batch_size, 16, 8, 8)
        # Tanh activation continues feature extraction.
        x = torch.tanh(self.conv2(x))

        # Apply second max pooling (batch_size, 16, 8, 8) -> (batch_size, 16, 4, 4)
        # Downsamples to focus on dominant features.
        x = self.pool2(x)

        # Flatten the tensor (batch_size, 16, 4, 4) -> (batch_size, 256)
        # Prepares for fully connected layers with 16*4*4=256 elements.
        x = x.view(-1, 16 * 4 * 4)

        # First fully connected + tanh (batch_size, 256) -> (batch_size, 120)
        # Tanh refines the flattened features.
        x = torch.tanh(self.fc1(x))

        # Second fully connected + tanh (batch_size, 120) -> (batch_size, 84)
        # Tanh prepares features for final output.
        x = torch.tanh(self.fc2(x))

        # Output layer (batch_size, 84) -> (batch_size, 10)
        # Raw logits, no activation applied.
        x = self.fc3(x)

        return x

def build_model():
    model = LeNet5()
    return model

model = build_model().to(DEVICE)

We'll now pass an image step by step through the model to visualize how features are extracted. We'll use the test loader (which doesn't shuffle) for predictability:


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def show_images(images, n_columns=3, cmap='gray', scale=10):
    # Convert PyTorch tensor to NumPy array if necessary, detaching from GPU/computation graph.
    # Ensures compatibility with matplotlib, which expects NumPy arrays.
    if isinstance(images, torch.Tensor):
        images = images.detach().cpu().numpy()

    # Determine number of images to display.
    # If images is 3D (e.g., NCHW), use first dim; if list/array of 2D images, use length.
    n_images = images.shape[0] if images.ndim == 3 else len(images)

    # Adjust n_columns: use the smaller of specified n_columns or n_images.
    # Prevents unused columns when there are fewer images than specified columns.
    n_columns = min(n_columns, n_images)

    # Extract height and width of images.
    # For 3D array (e.g., NCHW), take last two dims; for list of 2D arrays, use first image's shape.
    height, width = images.shape[-2:] if images.ndim == 3 else images[0].shape

    # Calculate number of rows needed based on adjusted n_columns.
    # Ceiling division ensures enough rows (e.g., 5 images, 3 cols -> 2 rows).
    n_rows = (n_images + n_columns - 1) // n_columns

    # Create a subplot grid with specified size.
    # Figure size scales with image dimensions and 'scale' factor; dpi=100 for clarity.
    fig, axes = plt.subplots(n_rows, n_columns,
                            figsize=(n_columns * width / 100 * scale, n_rows * height / 100 * scale),
                            dpi=100)

    # Flatten axes array for easy indexing if grid is multi-row/column.
    # If single row and column, wrap in list to maintain consistency.
    axes = axes.flatten() if n_rows > 1 or n_columns > 1 else [axes]

    # Loop through each image to display it.
    for i in range(n_images):
        ax = axes[i]  # Select the corresponding subplot.
        # Extract the i-th image: if 3D, use index i; if list, use i-th element.
        img = images[i] if images.ndim == 3 else images[i]
        ax.imshow(img, cmap=cmap)  # Display image with specified colormap (e.g., 'gray').
        ax.axis('off')  # Hide axes for cleaner visualization.

    # Turn off unused subplots if n_images < total grid slots.
    # Handles cases where grid size exceeds number of images.
    for i in range(n_images, len(axes)):
        axes[i].axis('off')

    # Adjust layout to minimize overlap and wasted space.
    plt.tight_layout()

    # Render the plot.
    plt.show()

images, labels = next(iter(test_loader))
image, label = images[0], labels[0]
image.shape, label, show_images(image)

Let's inspect the filters in the first convolutional layer (`conv1`):


In [ ]:
import math

def show_conv_filters(conv_layer, cmap='gray', scale=30):
    # Extract the weights from the conv layer
    weights = conv_layer.weight.data  # Shape: [out_channels, in_channels, height, width]

    # Get dimensions
    out_channels = weights.shape[0]  # Number of filters (columns)
    in_channels = weights.shape[1]   # Number of input channels per filter (rows per column)
    filter_height = weights.shape[2]
    filter_width = weights.shape[3]

    # Total number of items to display
    total_items = out_channels * in_channels

    # Reshape weights to [in_channels, out_channels, height, width] for column-wise layout
    weights_reshaped = weights.permute(1, 0, 2, 3)  # [in_channels, out_channels, height, width]
    weights_reshaped = weights_reshaped.reshape(total_items, filter_height, filter_width)
    # Shape: [in_channels * out_channels, height, width]

    # Normalize each 2D kernel to [0, 1] based on its own min and max
    normalized_weights = torch.zeros_like(weights_reshaped)
    for i in range(weights_reshaped.shape[0]):
        kernel = weights_reshaped[i]  # Shape: [height, width]
        kernel_min = kernel.min()
        kernel_max = kernel.max()
        if kernel_max > kernel_min: normalized_weights[i] = (kernel - kernel_min) / (kernel_max - kernel_min)
        else: normalized_weights[i] = kernel  # If max == min, leave as is

    # Set number of columns to the number of filters
    n_columns = out_channels

    # Print some info for context
    print(f"Weight shape: {weights.shape}")

    # Call show_images once with the normalized 3D tensor and adjusted scale
    show_images(normalized_weights, n_columns=n_columns, cmap=cmap, scale=scale)

show_conv_filters(model.conv1, scale=29)

The `conv1` layer has shape `(6, 1, 5, 5)`, meaning it applies 6 filters of size 5x5 to the input image, producing 6 feature maps. Let's confirm this by running an image through it:


In [ ]:
with torch.no_grad():
    _image1 = model.conv1(image.unsqueeze(0).to(DEVICE))  # Add batch dimension and move to device
_image1.shape, show_images(_image1.squeeze(0), n_columns=_image1.shape[1], scale=6)

Passing an image of shape `(1, 28, 28)` through `conv1` (shape `(6, 1, 5, 5)`) produces a tensor of shape `(6, 24, 24)`. This represents the 6 activation maps (each 24x24) generated by the 6 filters.


Next, we apply the nonlinear `tanh` function to the activation maps, squashing values into the range `[-1, 1]`:


In [ ]:
with torch.no_grad(): _image2 = torch.tanh(_image1)
_image2.shape, show_images(_image2.squeeze(0), n_columns=_image2.shape[1], scale=6)

Now, let's apply the max pooling layer:


In [ ]:
with torch.no_grad(): _image3 = model.pool1(_image2)
_image3.shape, show_images(_image3.squeeze(0), n_columns=_image3.shape[1], scale=12)

The max pooling layer reduces the activation maps from shape `(6, 24, 24)` to `(6, 12, 12)`, decreasing dimensionality.


Next, we'll pass these through the `conv2` layer. Let's inspect its filters:


In [ ]:
show_conv_filters(model.conv2, scale=20)

The `conv2` layer has shape `(16, 6, 5, 5)`, meaning it applies 16 filters (each spanning all 6 input channels) to the input of shape `(6, 12, 12)`. This should produce 16 activation maps of smaller size. Let's confirm:


In [ ]:
with torch.no_grad(): _image4 = model.conv2(_image3)
_image4.shape, show_images(_image4.squeeze(0), n_columns=_image4.shape[1], scale=10)

Passing the activation maps of shape `(6, 12, 12)` through `conv2` (shape `(16, 6, 5, 5)`) results in a tensor of shape `(16, 8, 8)`.


Now, apply the `tanh` activation:


In [ ]:
with torch.no_grad(): _image5 = torch.tanh(_image4)
_image5.shape, show_images(_image5.squeeze(0), n_columns=_image5.shape[1], scale=10)

Finally, apply the last max pooling layer:


In [ ]:
with torch.no_grad(): _image6 = model.pool2(_image5)
_image6.shape, show_images(_image6.squeeze(0), n_columns=_image6.shape[1], scale=20)

That's it! These activations will be flattened and passed through an MLP for classification. Up to this point, the network has learned how strongly each filter's pattern is present at each location in the image. The filters are still untrained, but we'll revisit them after training.


## Evaluate Model

In [ ]:
import torch.nn.functional as F

def evaluate(model, data_loader):
    # Set model in evaluation mode
    # (eg: disable dropout, change
    # batch norm statistics, etc.)
    model.eval()

    # Create the loss function
    loss_fn = nn.CrossEntropyLoss()

    # Evaluate model performance
    # predicting data from dataloader
    correct, total = 0, 0
    batch_losses = []
    with torch.no_grad():
        # Iterate through loader, one batch at a time
        for images, labels in data_loader:
            # Load batch into device (eg: GPU)
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            # Forward pass image batch through
            # device to get predictions
            logits = model(images)

            # Calculate loss (CrossEntropyLoss expects class indices, not one-hot)
            loss = loss_fn(logits, labels)
            batch_losses.append(loss.item())

            # Get predicted classes
            _, predicted = torch.max(logits.data, 1)

            # Count number of correct predictions
            batch_size = labels.size(0)
            total += batch_size
            correct += (predicted == labels).sum().item()

    # Calculate and return evaluation stats
    accuracy = 100 * correct / total
    loss = np.array(batch_losses).mean()
    return {
        "loss" : loss,
        "correct" : correct,
        "total" : total,
        "accuracy" : accuracy
    }

evaluate(model, test_loader)

## Train Model

In [ ]:
import numpy as np
from tqdm import tqdm
import torch.optim as optim

def train(
    model,
    train_loader,
    val_loader,
    n_epochs=None,
    learning_rate=None,
    eval_frequency=None
):
    if n_epochs is None: n_epochs = CONFIG["n_epochs"]
    if learning_rate is None: learning_rate = CONFIG["learning_rate"]

    # Set model in training mode
    model.train()

    # Initialize optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    # Initialize training history
    history = {
        "train_losses": [],
        "val_losses": [],
        "val_accuracies": []
    }

    for epoch in range(n_epochs):
        batch_losses = []

        # Create tqdm instance manually (no with statement)
        dataloader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}")
        total_batches = len(dataloader_tqdm)

        # Training loop
        for i, (images, labels) in enumerate(dataloader_tqdm):
            # Move data to device
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            # Forward pass
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, labels)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # Track losses
            train_loss_step = loss.item()
            batch_losses.append(train_loss_step)

            dataloader_tqdm.set_postfix({
                "train_loss_step": f"{train_loss_step:.3f}"
            })

            # If this is not the last iteration then move to next batch
            # (we'll evaluate on the test set after the last batch is done)
            last_iteration = (i + 1) == total_batches
            if not last_iteration: continue

            # After training loop, calculate epoch loss
            train_loss_epoch = np.array(batch_losses).mean()
            history["train_losses"].append(train_loss_epoch)

            # Run evaluation on validation set
            eval_results = evaluate(model, val_loader)
            val_loss = eval_results["loss"]
            val_accuracy = eval_results["accuracy"]
            history["val_losses"].append(val_loss)
            history["val_accuracies"].append(val_accuracy)

            # Update tqdm postfix with epoch and validation results before closing
            dataloader_tqdm.set_postfix({
                "train_loss_step": f"{train_loss_step:.3f}",
                "train_loss_epoch": f"{train_loss_epoch:.3f}",
                "val_loss": f"{val_loss:.3f}",
                "val_acc": f"{val_accuracy:.3f}"
            })

    return history

_ = train(model, train_loader, val_loader=test_loader)

And we're done! After 5 epochs, we've achieved over 98% classification accuracy. This can be further improved by:

- Using ReLU activations instead of tanh
- Adding batch normalization layers
- Incorporating dropout
- Making the model deeper
- Adding residual connections
- Augmenting the dataset


Take another look at the filters in the first convolutional layer and compare them to the untrained ones. Notice how the filters are now less random and focus on different edges and patterns.


In [ ]:
show_conv_filters(model.conv1, scale=29)